In [1]:
import requests
from bs4 import BeautifulSoup
import json
import pandas as pd
import time
import os
from pathlib import Path

In [29]:
current_directory = os.getcwd()
csv_name = os.path.join(current_directory, "..", "data", "prod_urls.csv")
prod_url = pd.read_csv(csv_name)
prod_url.head(5)

,product_id,url,status,discovered_at
0,698859,https://www.naiin.com/product/detail/698859/,pending,2026-03-15 16:23:55.176126
1,698857,https://www.naiin.com/product/detail/698857/,pending,2026-03-15 16:23:55.176126
2,698856,https://www.naiin.com/product/detail/698856/,pending,2026-03-15 16:23:55.176126
3,699421,https://www.naiin.com/product/detail/699421/,pending,2026-03-15 16:23:55.176126
4,698858,https://www.naiin.com/product/detail/698858/,pending,2026-03-15 16:23:55.176126


In [30]:
prod_url = prod_url.drop(columns=['status', 'discovered_at'])
prod_url['product_id'] = prod_url['product_id'].astype(int)
prod_url.head(5)

,product_id,url
0,698859,https://www.naiin.com/product/detail/698859/
1,698857,https://www.naiin.com/product/detail/698857/
2,698856,https://www.naiin.com/product/detail/698856/
3,699421,https://www.naiin.com/product/detail/699421/
4,698858,https://www.naiin.com/product/detail/698858/


In [4]:
examples = prod_url.head(5)
examples

,product_id,url
0,698859,https://www.naiin.com/product/detail/698859/
1,698857,https://www.naiin.com/product/detail/698857/
2,698856,https://www.naiin.com/product/detail/698856/
3,699421,https://www.naiin.com/product/detail/699421/
4,698858,https://www.naiin.com/product/detail/698858/


In [2]:
import re
import json
import html

def extract_extra_info(html_content):
    extra_data = {}
    # ใช้ BeautifulSoup ช่วยสำหรับส่วนที่ JSON ไม่มีข้อมูล
    soup = BeautifulSoup(html_content, 'lxml')
    
    try:
        # 1. ดึง Rating และข้อมูลเบื้องต้นจาก JSON (วิธีเดิมที่คุณใช้แล้วได้ผล)
        match = re.search(r'AverageRating&quot;:(.+?)\}', html_content)
        if match:
            raw_json = '{"AverageRating":' + match.group(1) + '}'
            clean_json = html.unescape(raw_json)
            json_obj = json.loads(clean_json)
            
            extra_data = {
                'AverageRating': json_obj.get('AverageRating') or None,
                'TotalRating': json_obj.get('TotalRating') or None,
                'NumberOfPage': json_obj.get('NumberOfPage') or None,
                # เริ่มต้นด้วย None เดี๋ยวเราใช้ BeautifulSoup ทับถ้าหาเจอ
                'Width': None,
                'Height': None,
                'Thick': None,
                'GrossWeightKG': None,
                'FileSizeMB': None
            }

        # 2. ใช้ BeautifulSoup เจาะหา "ขนาดไฟล์", "ขนาด", "น้ำหนัก" จาก Label โดยตรง
        # เราจะหา <label> ที่มีข้อความที่ต้องการ แล้วหา <p> ที่อยู่ถัดจากมัน
        labels = soup.find_all('label', class_='product-label')
        for label in labels:
            text = label.get_text(strip=True)
            detail_p = label.find_next_sibling('p', class_='product-label-detail')
            
            if detail_p:
                val = detail_p.get_text(strip=True)
                
                if "ขนาดไฟล์" in text:
                    # ดึงเฉพาะตัวเลขจาก "4.08 MB"
                    size_val = re.search(r'[\d\.]+', val)
                    if size_val: extra_data['FileSizeMB'] = size_val.group()
                
                elif "ขนาด" in text:
                    # แยก "0 x 0 x 0 CM" ออกเป็น 3 ส่วน
                    dims = re.findall(r'[\d\.]+', val)
                    if len(dims) >= 3:
                        extra_data['Width'] = dims[0]
                        extra_data['Height'] = dims[1]
                        extra_data['Thick'] = dims[2]
                
                elif "น้ำหนัก" in text:
                    # ดึงเฉพาะตัวเลขจาก "0.374 KG"
                    weight_val = re.search(r'[\d\.]+', val)
                    if weight_val: extra_data['GrossWeightKG'] = weight_val.group()

    except Exception as e:
        print(f"  [Warning] Extra info skip: {e}")
        
    return extra_data

In [37]:
def scrape(url):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Referer': 'https://www.naiin.com/',
    }
    
    time.sleep(1.5) # เพิ่มเวลาหน่อยกันโดนบล็อก
    response = requests.get(url, headers=headers, timeout=10)
    html_text = response.text
    soup = BeautifulSoup(response.text, 'lxml')
    
    # ข้อมูลพื้นฐานจาก Metadata
    # 1. วิธีเดิม (Fast & Native) สำหรับฟิลด์ที่ไม่มีปัญหา
    meta_map = {
        'Title': soup.find('meta', property='og:title'),
        'Price_Full': soup.find('meta', property='og:product:price:amount'),
        'Barcode': soup.find('meta', property='book:isbn'),
        'Release_Date': soup.find('meta', property='book:release_date'),
    }
    
    data = {'url': url}
    for key, tag in meta_map.items():
        data[key] = tag['content'].strip() if tag else "N/A"

# ใช้ Regex ที่มองหาจุดสิ้นสุดคือ "> หรือ /> (จุดปิดแท็ก meta)
# โดยเก็บทุกอย่าง (.*) ที่อยู่ระหว่าง content=" จนถึงจุดนั้น
    kw_pattern = r'<meta[^>]*name="keywords"[^>]*content="(.*)"\s*/?>'
    kw_match = re.search(kw_pattern, html_text, re.IGNORECASE | re.DOTALL)

    if kw_match:
        # raw_content จะได้ข้อมูลมาจนถึง " ตัวสุดท้ายก่อนปิดแท็ก
        raw_content = kw_match.group(1).strip()
        
        # เนื่องจากเราใช้ (.*) แบบ Greedy มันอาจจะกินพื้นที่ไปจนถึงแท็กอื่นที่ปิดด้วย "> เหมือนกัน
        # เราจะแก้ด้วยการเอาสตริงที่ได้มาตัด (split) ด้วย "> อีกรอบเพื่อความชัวร์
        clean_content = raw_content.split('">')[0]
        
        # สุดท้าย ถ้ามีเครื่องหมาย " ปิดท้ายสตริงที่เหลืออยู่ (ซึ่งเป็นตัวปิดของ content) ให้เอาออก
        data['keywords'] = clean_content.rstrip('"')
    else:
        data['keywords'] = "N/A"
    # --- ส่วนที่แก้ปัญหา Error ---
    # ส่ง response.text (String) เข้าไป
    extra_info = extract_extra_info(response.text)
    
    # รวมข้อมูลเข้าด้วยกัน
    data.update(extra_info)
    
    return data

In [38]:
scrape('https://www.naiin.com/product/detail/691950/')

{'url': 'https://www.naiin.com/product/detail/691950/',
 'Title': 'เสด็จสู่สวรรคาลัย "ราชินีแห่งราชัน" คู่พระบารมีในหลวงรัชกาลที่ ๙Books | ร้านหนังสือนายอินทร์',
 'Price_Full': '195',
 'Barcode': '9786166092776',
 'Release_Date': '2025-12-19',
 'keywords': 'เสด็จสู่สวรรคาลัย " ราชินีแห่งราชัน" คู่พระบารมีในหลวงรัชกาลที่ ๙, พินิจ จันทร และคณะ, บันทึกสยาม, หนังสือพระราชนิพนธ์, หนังสือพระราชประวัติราชวงศ์, เสด็จสู่สวรรคาลัย,ราชินีแห่งราชัน,สมเด็จพระนางเจ้าสิริกิติ์',
 'AverageRating': None,
 'TotalRating': None,
 'NumberOfPage': 160,
 'Width': '16.5',
 'Height': '23.9',
 'Thick': '0.9',
 'GrossWeightKG': '0.278',
 'FileSizeMB': None}